# IL-RL on Kaggle Dual T4: GRPO Training + ARC-AGI-3-Style Benchmark (v2 — fast)

## Goal

Load **DeepSeek-R1-Distill-Qwen-1.5B** on a Kaggle T4, train it with
**Group-Relative Policy Optimization (GRPO)** on ARC-AGI-3-style grid puzzles,
and benchmark it on **40 held-out transfer tasks** before and after RL.

PyTorch/CUDA port of the MLX-based ILresearch pipeline.

## v2 fixes (speed)
- **`model.generate()`** with KV cache instead of manual token loop (was O(n³), now O(n²))
- **Single GPU** (`cuda:0`) — a 1.5B model (3 GB bf16) fits on one T4; pipeline
  parallelism across 2 GPUs is *slower* for sequential token generation
- **Reduced token budgets** (think=128, pred=96) and **skipped in-training eval**
  (the pre/post 40-task benchmark is the real comparison)

> **Kaggle settings**: **GPU T4 x2** + **Internet on**. (Second T4 provides
> memory headroom; the 1.5B model runs on cuda:0 for max generation throughput.)


## 0. Install & Import

In [ ]:
# Uninstall preinstalled torchao (Kaggle ships 0.10, peft needs >0.16).
# We don't use torchao — removing it avoids the version-check ImportError.
!pip uninstall -y torchao >/dev/null 2>&1
!pip install -q transformers==4.46.3 accelerate peft


In [ ]:
import os, sys, re, json, time, random, traceback
from copy import deepcopy
from collections import deque

import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model, TaskType

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# ── Config (tuned for speed on T4) ──
MODEL_NAME        = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"
N_TRAIN_ITERS     = 40      # GRPO iterations
GROUP_SIZE        = 4       # rollouts per puzzle
N_STEPS           = 2       # feedback rounds per episode
THINKING_TOKENS   = 128     # reasoning tokens per step (reduced from 256)
PREDICTION_TOKENS = 96      # answer tokens per step (reduced from 128)
LR                = 5e-6
TEMPERATURE       = 0.8
TOP_P             = 0.9
GAMMA             = 0.9
KL_BETA           = 0.04
CLIP_EPS          = 0.2
LORA_RANK         = 8
LORA_LAYERS       = 16
BENCHMARK_SIZE    = 40
BENCH_THINK       = 256     # benchmark gets more thinking budget
BENCH_ANSWER      = 128

DEVICE = "cuda:0"
print(f"Config: {N_TRAIN_ITERS} iters, think={THINKING_TOKENS}, pred={PREDICTION_TOKENS}")
print(f"Benchmark: {BENCHMARK_SIZE} puzzles, think={BENCH_THINK}, pred={BENCH_ANSWER}")


## 1. Load Model on single T4 (cuda:0)

A 1.5B model in bfloat16 is ~3 GB — fits comfortably on one T4 (16 GB).
Loading on a single GPU avoids the pipeline-parallelism bubble that makes
sequential generation slow across two GPUs. The second T4 is available as
memory headroom.


In [ ]:
n_gpus = torch.cuda.device_count()
print(f"Available GPUs: {n_gpus}")
for i in range(n_gpus):
    props = torch.cuda.get_device_properties(i)
    print(f"  cuda:{i} — {props.name}, {props.total_memory / 1e9:.1f} GB")
assert n_gpus >= 1, "No GPU — enable T4 x2 in Kaggle settings."

print(f"\nLoading {MODEL_NAME} (bfloat16, {DEVICE}) ...")
t0 = time.time()
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map={"" : DEVICE},   # all on cuda:0
)
base_model.eval()
base_model.config.use_cache = True

n_params = sum(p.numel() for p in base_model.parameters())
print(f"Loaded in {time.time()-t0:.1f}s | params: {n_params/1e9:.2f}B")
print(f"Device: {next(base_model.parameters()).device}")


## 2. Apply LoRA (rank 8, last 16 layers)

In [ ]:
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=LORA_RANK,
    lora_alpha=LORA_RANK,
    lora_dropout=0.0,
    bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    layers_to_transform=list(range(28 - LORA_LAYERS, 28)),
)
model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()
trainable = [p for p in model.parameters() if p.requires_grad]
assert trainable, "No trainable parameters — LoRA setup failed."


## 3. Grid Utilities & Puzzle Generators

**20 training types** (for GRPO) + **12 transfer/benchmark types** (different —
tests transfer of intuition, not memorization).


In [ ]:
# ============================================================
# Grid Utilities
# ============================================================
def empty_grid(h, w, val=0): return [[val]*w for _ in range(h)]
def grid_copy(g): return [row[:] for row in g]
def grid_dims(g): return len(g), len(g[0])
def grid_to_str(g):
    rows = ['[' + ','.join(str(c) for c in row) + ']' for row in g]
    return '[' + ',\n '.join(rows) + ']'
def count_nonzero(g): return sum(1 for row in g for c in row if c != 0)
def distinct_colors(g): return set(c for row in g for c in row if c != 0)
def connected_components(g, connectivity=4):
    h, w = grid_dims(g); visited = empty_grid(h, w, False); comps = []
    nbrs = [(-1,0),(1,0),(0,-1),(0,1)] if connectivity==4 else \
           [(-1,0),(1,0),(0,-1),(0,1),(-1,-1),(-1,1),(1,-1),(1,1)]
    for r in range(h):
        for c in range(w):
            if g[r][c]!=0 and not visited[r][c]:
                color=g[r][c]; comp=[]; q=deque([(r,c)]); visited[r][c]=True
                while q:
                    cr,cc=q.popleft(); comp.append((cr,cc))
                    for dr,dc in nbrs:
                        nr,nc=cr+dr,cc+dc
                        if 0<=nr<h and 0<=nc<w and not visited[nr][nc] and g[nr][nc]==color:
                            visited[nr][nc]=True; q.append((nr,nc))
                comps.append({'color':color,'cells':comp,'size':len(comp)})
    return comps
def bounding_box(cells):
    rs=[r for r,c in cells]; cs=[c for r,c in cells]
    return min(rs),min(cs),max(rs),max(cs)
print("Grid utilities loaded.")


In [ ]:
# ============================================================
# TRAINING PUZZLE TYPES (20) — from il/environments.py
# ============================================================
def color_swap_params(rng): a,b=rng.sample(range(1,6),2); return {'a':a,'b':b}
def color_swap_instance(p,rng):
    h,w=rng.randint(4,7),rng.randint(4,7); g=empty_grid(h,w)
    for _ in range(rng.randint(4,h*w//2)): g[rng.randint(0,h-1)][rng.randint(0,w-1)]=rng.randint(1,5)
    out=grid_copy(g)
    for r in range(h):
        for c in range(w):
            if out[r][c]==p['a']: out[r][c]=p['b']
    return g,out
def color_swap_desc(p): return f"Every cell with color {p['a']} becomes color {p['b']}."

def rotate_params(rng): return {'angle':rng.choice([90,180,270])}
def rotate_instance(p,rng):
    h,w=rng.randint(4,7),rng.randint(4,7); g=empty_grid(h,w)
    for _ in range(rng.randint(5,h*w//2)): g[rng.randint(0,h-1)][rng.randint(0,w-1)]=rng.randint(1,5)
    a=p['angle']
    if a==90: out=[[g[h-1-r][c] for r in range(h)] for c in range(w)]
    elif a==180: out=[row[::-1] for row in g[::-1]]
    else: out=[[g[r][w-1-c] for r in range(h)] for c in range(w)][::-1]
    return g,out
def rotate_desc(p): return f"Rotate the grid {p['angle']} degrees."

def border_params(rng): return {'color':rng.randint(1,5)}
def border_instance(p,rng):
    h,w=rng.randint(3,6),rng.randint(3,6); g=empty_grid(h,w)
    for _ in range(rng.randint(2,h*w//3)): g[rng.randint(0,h-1)][rng.randint(0,w-1)]=rng.randint(1,5)
    out=empty_grid(h+2,w+2,0)
    for r in range(h):
        for c in range(w): out[r+1][c+1]=g[r][c]
    for c in range(w+2): out[0][c]=p['color']; out[h+1][c]=p['color']
    for r in range(h+2): out[r][0]=p['color']; out[r][w+1]=p['color']
    return g,out
def border_desc(p): return f"Add a border of color {p['color']} around the grid."

def interior_fill_params(rng): return {'color':rng.randint(1,5)}
def interior_fill_instance(p,rng):
    h,w=rng.randint(5,8),rng.randint(5,8); g=empty_grid(h,w)
    for _ in range(rng.randint(2,4)):
        if rng.random()<0.5:
            r=rng.randint(1,h-2); c1,c2=sorted(rng.sample(range(w),2))
            for c in range(c1,c2+1): g[r][c]=1
        else:
            c=rng.randint(1,w-2); r1,r2=sorted(rng.sample(range(h),2))
            for r in range(r1,r2+1): g[r][c]=1
    visited=empty_grid(h,w,False); q=deque()
    for r in range(h):
        for c in range(w):
            if (r==0 or r==h-1 or c==0 or c==w-1) and g[r][c]==0: q.append((r,c)); visited[r][c]=True
    while q:
        r,c=q.popleft()
        for dr,dc in [(-1,0),(1,0),(0,-1),(0,1)]:
            nr,nc=r+dr,c+dc
            if 0<=nr<h and 0<=nc<w and not visited[nr][nc] and g[nr][nc]==0: visited[nr][nc]=True; q.append((nr,nc))
    out=grid_copy(g)
    for r in range(h):
        for c in range(w):
            if g[r][c]==0 and not visited[r][c]: out[r][c]=p['color']
    return g,out
def interior_fill_desc(p): return f"Fill enclosed empty regions with color {p['color']}."

def row_shift_params(rng): return {}
def row_shift_instance(p,rng):
    h,w=rng.randint(4,7),rng.randint(4,7); g=empty_grid(h,w)
    for _ in range(rng.randint(4,h*w//2)): g[rng.randint(0,h-1)][rng.randint(0,w-1)]=rng.randint(1,5)
    out=empty_grid(h,w)
    for r in range(h):
        s=r%w
        for c in range(w): out[r][(c+s)%w]=g[r][c]
    return g,out
def row_shift_desc(p): return "Shift each row right by its row index (mod width)."

def mirror_half_params(rng): return {'axis':rng.choice(['h','v'])}
def mirror_half_instance(p,rng):
    h,w=rng.randint(4,8),rng.randint(4,8); g=empty_grid(h,w)
    for _ in range(rng.randint(5,h*w//2)): g[rng.randint(0,h-1)][rng.randint(0,w-1)]=rng.randint(1,5)
    out=grid_copy(g)
    if p['axis']=='h':
        for r in range(h):
            for c in range(w//2): out[r][w-1-c]=out[r][c]
    else:
        for r in range(h//2):
            for c in range(w): out[h-1-r][c]=out[r][c]
    return g,out
def mirror_half_desc(p): return "Mirror left half to right half." if p['axis']=='h' else "Mirror top half to bottom half."

def crop_params(rng): return {}
def crop_instance(p,rng):
    h,w=rng.randint(6,9),rng.randint(6,9); g=empty_grid(h,w)
    ih,iw=rng.randint(2,h-2),rng.randint(2,w-2); r0,c0=rng.randint(0,h-ih),rng.randint(0,w-iw)
    for r in range(ih):
        for c in range(iw):
            if rng.random()<0.5: g[r0+r][c0+c]=rng.randint(1,5)
    nz=[(r,c) for r in range(h) for c in range(w) if g[r][c]!=0]
    if not nz: g[rng.randint(0,h-1)][rng.randint(0,w-1)]=2; nz=[(r,c) for r in range(h) for c in range(w) if g[r][c]!=0]
    rs=[r for r,c in nz]; cs=[c for r,c in nz]
    out=[[g[r][c] for c in range(min(cs),max(cs)+1)] for r in range(min(rs),max(rs)+1)]
    return g,out
def crop_desc(p): return "Crop to bounding box of non-zero cells."

def scale_params(rng): return {'factor':rng.randint(2,3)}
def scale_instance(p,rng):
    h,w=rng.randint(3,5),rng.randint(3,5); g=empty_grid(h,w)
    for _ in range(rng.randint(3,h*w//2)): g[rng.randint(0,h-1)][rng.randint(0,w-1)]=rng.randint(1,5)
    f=p['factor']; out=empty_grid(h*f,w*f)
    for r in range(h):
        for c in range(w):
            v=g[r][c]
            for dr in range(f):
                for dc in range(f): out[r*f+dr][c*f+dc]=v
    return g,out
def scale_desc(p): return f"Scale up {p['factor']}x — each cell becomes a {p['factor']}x{p['factor']} block."

def adjacency_params(rng): return {}
def adjacency_instance(p,rng):
    h,w=rng.randint(5,8),rng.randint(5,8); g=empty_grid(h,w)
    for _ in range(rng.randint(5,h*w//3)): g[rng.randint(0,h-1)][rng.randint(0,w-1)]=rng.randint(1,5)
    out=empty_grid(h,w)
    for r in range(h):
        for c in range(w):
            if g[r][c]==0:
                n=0
                for dr,dc in [(-1,0),(1,0),(0,-1),(0,1)]:
                    nr,nc=r+dr,c+dc
                    if 0<=nr<h and 0<=nc<w and g[nr][nc]!=0: n+=1
                out[r][c]=min(n,9)
            else: out[r][c]=g[r][c]
    return g,out
def adjacency_desc(p): return "Empty cells become the count of their non-zero neighbors."

def threshold_params(rng): return {'threshold':rng.randint(2,4)}
def threshold_instance(p,rng):
    h,w=rng.randint(5,8),rng.randint(5,8); g=empty_grid(h,w)
    for _ in range(rng.randint(6,h*w//2)): g[rng.randint(0,h-1)][rng.randint(0,w-1)]=rng.randint(1,5)
    t=p['threshold']; out=empty_grid(h,w)
    for r in range(h):
        for c in range(w):
            if g[r][c]>=t: out[r][c]=g[r][c]
    return g,out
def threshold_desc(p): return f"Keep cells >= {p['threshold']}, others become 0."

def erosion_params(rng): return {}
def erosion_instance(p,rng):
    h,w=rng.randint(5,8),rng.randint(5,8); g=empty_grid(h,w)
    for _ in range(rng.randint(4,10)):
        color=rng.randint(1,5); r0,c0=rng.randint(0,h-1),rng.randint(0,w-1)
        for _ in range(rng.randint(2,5)):
            dr,dc=rng.choice([(-1,0),(1,0),(0,-1),(0,1)]); nr,nc=r0+dr,c0+dc
            if 0<=nr<h and 0<=nc<w: g[nr][nc]=color; r0,c0=nr,nc
    out=grid_copy(g)
    for r in range(h):
        for c in range(w):
            if g[r][c]!=0:
                b=False
                for dr,dc in [(-1,0),(1,0),(0,-1),(0,1)]:
                    nr,nc=r+dr,c+dc
                    if not(0<=nr<h and 0<=nc<w) or g[nr][nc]!=g[r][c]: b=True; break
                if b: out[r][c]=0
    return g,out
def erosion_desc(p): return "Remove outermost cells of each object (erosion)."

def dilation_params(rng): return {}
def dilation_instance(p,rng):
    h,w=rng.randint(5,8),rng.randint(5,8); g=empty_grid(h,w)
    for _ in range(rng.randint(3,8)): g[rng.randint(0,h-1)][rng.randint(0,w-1)]=rng.randint(1,5)
    out=grid_copy(g)
    for r in range(h):
        for c in range(w):
            if g[r][c]!=0:
                for dr,dc in [(-1,0),(1,0),(0,-1),(0,1)]:
                    nr,nc=r+dr,c+dc
                    if 0<=nr<h and 0<=nc<w and out[nr][nc]==0: out[nr][nc]=g[r][c]
    return g,out
def dilation_desc(p): return "Expand each object by 1 cell in all directions."

def transpose_params(rng): return {}
def transpose_instance(p,rng):
    h,w=rng.randint(4,7),rng.randint(4,7); g=empty_grid(h,w)
    for _ in range(rng.randint(5,h*w//2)): g[rng.randint(0,h-1)][rng.randint(0,w-1)]=rng.randint(1,5)
    out=[[g[r][c] for r in range(h)] for c in range(w)]
    return g,out
def transpose_desc(p): return "Transpose — swap rows and columns."

def center_extract_params(rng): return {'size':rng.randint(2,4)}
def center_extract_instance(p,rng):
    h,w=rng.randint(6,9),rng.randint(6,9); g=empty_grid(h,w)
    for _ in range(rng.randint(8,h*w//2)): g[rng.randint(0,h-1)][rng.randint(0,w-1)]=rng.randint(1,5)
    s=p['size']; r0=(h-s)//2; c0=(w-s)//2
    out=[[g[r0+r][c0+c] for c in range(s)] for r in range(s)]
    return g,out
def center_extract_desc(p): return f"Extract center {p['size']}x{p['size']} region."

def color_pos_params(rng): return {'parity':rng.choice(['even_col','odd_col','even_row','odd_row'])}
def color_pos_instance(p,rng):
    h,w=rng.randint(4,7),rng.randint(4,7); g=empty_grid(h,w)
    for _ in range(rng.randint(5,h*w//2)): g[rng.randint(0,h-1)][rng.randint(0,w-1)]=rng.randint(1,4)
    out=grid_copy(g); par=p['parity']
    for r in range(h):
        for c in range(w):
            if g[r][c]!=0:
                if par=='even_col' and c%2==0: out[r][c]=min(g[r][c]+1,9)
                elif par=='odd_col' and c%2==1: out[r][c]=min(g[r][c]+1,9)
                elif par=='even_row' and r%2==0: out[r][c]=min(g[r][c]+1,9)
                elif par=='odd_row' and r%2==1: out[r][c]=min(g[r][c]+1,9)
    return g,out
def color_pos_desc(p): return f"Non-zero cells in {p['parity'].replace('_',' ')} get +1 color."

def outline_params(rng): return {}
def outline_instance(p,rng):
    h,w=rng.randint(5,8),rng.randint(5,8); g=empty_grid(h,w)
    for _ in range(rng.randint(2,5)):
        color=rng.randint(1,4); r0,c0=rng.randint(1,h-2),rng.randint(1,w-2)
        for dr in range(rng.randint(1,3)):
            for dc in range(rng.randint(1,3)):
                if 0<=r0+dr<h and 0<=c0+dc<w: g[r0+dr][c0+dc]=color
    out=grid_copy(g)
    for r in range(h):
        for c in range(w):
            if g[r][c]==0:
                for dr,dc in [(-1,0),(1,0),(0,-1),(0,1)]:
                    nr,nc=r+dr,c+dc
                    if 0<=nr<h and 0<=nc<w and g[nr][nc]!=0: out[r][c]=9; break
    return g,out
def outline_desc(p): return "Draw outline (color 9) around each object."

def max_row_params(rng): return {}
def max_row_instance(p,rng):
    h,w=rng.randint(4,7),rng.randint(4,7); g=empty_grid(h,w)
    for _ in range(rng.randint(5,h*w//2)): g[rng.randint(0,h-1)][rng.randint(0,w-1)]=rng.randint(1,5)
    out=empty_grid(h,w)
    for r in range(h):
        mx=max(g[r])
        if mx>0: out[r]=[mx]*w
    return g,out
def max_row_desc(p): return "Fill each row with its max value."

def flip_color_params(rng): return {'color':rng.randint(1,5)}
def flip_color_instance(p,rng):
    h,w=rng.randint(4,7),rng.randint(4,7); g=empty_grid(h,w)
    for _ in range(rng.randint(5,h*w//2)): g[rng.randint(0,h-1)][rng.randint(0,w-1)]=rng.randint(1,5)
    col=p['color']; targets=[(r,c) for r in range(h) for c in range(w) if g[r][c]==col]
    if len(targets)<=1:
        for _ in range(2): r,c=rng.randint(0,h-1),rng.randint(0,w-1); g[r][c]=col
        targets=[(r,c) for r in range(h) for c in range(w) if g[r][c]==col]
    out=grid_copy(g)
    for r,c in targets: out[r][c]=0
    for i,(r,c) in enumerate(targets): r2,c2=targets[len(targets)-1-i]; out[r2][c2]=col
    return g,out
def flip_color_desc(p): return f"Reverse positions of color {p['color']} cells."

def quadrant_params(rng): return {}
def quadrant_instance(p,rng):
    s=rng.choice([4,6,8]); h=w=s; g=empty_grid(h,w)
    for _ in range(rng.randint(6,h*w//2)): g[rng.randint(0,h-1)][rng.randint(0,w-1)]=rng.randint(1,5)
    out=empty_grid(h,w); mid=h//2
    for r in range(mid):
        for c in range(mid):
            out[r][c]=g[r+mid][c+mid]; out[r+mid][c+mid]=g[r][c]
            out[r][c+mid]=g[r+mid][c]; out[r+mid][c]=g[r][c+mid]
    return g,out
def quadrant_desc(p): return "Swap quadrants diagonally."

def col_reverse_params(rng): return {}
def col_reverse_instance(p,rng):
    h,w=rng.randint(4,7),rng.randint(4,7); g=empty_grid(h,w)
    for _ in range(rng.randint(5,h*w//2)): g[rng.randint(0,h-1)][rng.randint(0,w-1)]=rng.randint(1,5)
    out=[row[::-1] for row in g]
    return g,out
def col_reverse_desc(p): return "Reverse the order of columns."

TRAIN_TYPES = [
    {'name':'color_swap','desc':'Color Swap','gen_params':color_swap_params,'gen_instance':color_swap_instance,'describe':color_swap_desc},
    {'name':'rotate','desc':'Grid Rotation','gen_params':rotate_params,'gen_instance':rotate_instance,'describe':rotate_desc},
    {'name':'border','desc':'Border Addition','gen_params':border_params,'gen_instance':border_instance,'describe':border_desc},
    {'name':'interior_fill','desc':'Interior Fill','gen_params':interior_fill_params,'gen_instance':interior_fill_instance,'describe':interior_fill_desc},
    {'name':'row_shift','desc':'Row Shift','gen_params':row_shift_params,'gen_instance':row_shift_instance,'describe':row_shift_desc},
    {'name':'mirror_half','desc':'Mirror Half','gen_params':mirror_half_params,'gen_instance':mirror_half_instance,'describe':mirror_half_desc},
    {'name':'crop','desc':'Crop to Content','gen_params':crop_params,'gen_instance':crop_instance,'describe':crop_desc},
    {'name':'scale','desc':'Scale Up','gen_params':scale_params,'gen_instance':scale_instance,'describe':scale_desc},
    {'name':'adjacency','desc':'Adjacency Coloring','gen_params':adjacency_params,'gen_instance':adjacency_instance,'describe':adjacency_desc},
    {'name':'threshold','desc':'Threshold Filter','gen_params':threshold_params,'gen_instance':threshold_instance,'describe':threshold_desc},
    {'name':'erosion','desc':'Erosion','gen_params':erosion_params,'gen_instance':erosion_instance,'describe':erosion_desc},
    {'name':'dilation','desc':'Dilation','gen_params':dilation_params,'gen_instance':dilation_instance,'describe':dilation_desc},
    {'name':'transpose','desc':'Grid Transpose','gen_params':transpose_params,'gen_instance':transpose_instance,'describe':transpose_desc},
    {'name':'center_extract','desc':'Center Extraction','gen_params':center_extract_params,'gen_instance':center_extract_instance,'describe':center_extract_desc},
    {'name':'color_pos','desc':'Color by Position','gen_params':color_pos_params,'gen_instance':color_pos_instance,'describe':color_pos_desc},
    {'name':'outline','desc':'Object Outline','gen_params':outline_params,'gen_instance':outline_instance,'describe':outline_desc},
    {'name':'max_row','desc':'Max per Row','gen_params':max_row_params,'gen_instance':max_row_instance,'describe':max_row_desc},
    {'name':'flip_color','desc':'Flip Color Positions','gen_params':flip_color_params,'gen_instance':flip_color_instance,'describe':flip_color_desc},
    {'name':'quadrant','desc':'Quadrant Swap','gen_params':quadrant_params,'gen_instance':quadrant_instance,'describe':quadrant_desc},
    {'name':'col_reverse','desc':'Column Reverse','gen_params':col_reverse_params,'gen_instance':col_reverse_instance,'describe':col_reverse_desc},
]
print(f"Loaded {len(TRAIN_TYPES)} TRAINING puzzle types.")


In [ ]:
# ============================================================
# BENCHMARK / TRANSFER PUZZLE TYPES (12) — from mini_arc_agi3_benchmark.ipynb
# INTENTIONALLY DIFFERENT from training types.
# ============================================================
def gravity_sort_params(rng): return {}
def gravity_sort_instance(p,rng):
    h,w=rng.randint(5,8),rng.randint(5,8); g=empty_grid(h,w)
    for _ in range(rng.randint(5,min(h*w//3,14))): g[rng.randint(0,h-1)][rng.randint(0,w-1)]=rng.randint(1,4)
    fallen=empty_grid(h,w)
    for c in range(w):
        col=[g[r][c] for r in range(h) if g[r][c]!=0]
        for i,v in enumerate(col): fallen[h-len(col)+i][c]=v
    cols=[]
    for c in range(w):
        col=[fallen[r][c] for r in range(h)]; bottom=next((v for v in reversed(col) if v!=0),0)
        cols.append((bottom,c,col))
    cols.sort(key=lambda x:(x[0],x[1])); out=empty_grid(h,w)
    for nc,(_,_,col) in enumerate(cols):
        for r in range(h): out[r][nc]=col[r]
    return g,out

def maze_path_params(rng): return {}
def maze_path_instance(p,rng):
    for _ in range(50):
        h,w=rng.randint(6,9),rng.randint(6,9); g=empty_grid(h,w,0)
        for _ in range(rng.randint(h*w//4,h*w//3)): g[rng.randint(0,h-1)][rng.randint(0,w-1)]=1
        g[0][0]=2; g[h-1][w-1]=3
        for dr,dc in [(-1,0),(1,0),(0,-1),(0,1)]:
            for r,c in [(0+dr,0+dc),(h-1+dr,w-1+dc)]:
                if 0<=r<h and 0<=c<w and g[r][c]==1: g[r][c]=0
        q=deque([(0,0)]); visited={(0,0)}; parent={(0,0):None}; found=False
        while q:
            r,c=q.popleft()
            if (r,c)==(h-1,w-1): found=True; break
            for dr,dc in [(-1,0),(1,0),(0,-1),(0,1)]:
                nr,nc=r+dr,c+dc
                if 0<=nr<h and 0<=nc<w and (nr,nc) not in visited and g[nr][nc]!=1:
                    visited.add((nr,nc)); parent[(nr,nc)]=(r,c); q.append((nr,nc))
        if not found: continue
        path=[]; cur=(h-1,w-1)
        while cur is not None: path.append(cur); cur=parent[cur]
        path.reverse(); out=grid_copy(g)
        for r,c in path[1:-1]: out[r][c]=4
        return g,out
    h=w=6; g=empty_grid(h,w,0); g[0][0]=2; g[5][5]=3; out=grid_copy(g)
    for i in range(1,5): out[0][i]=4
    for i in range(1,5): out[i][5]=4
    return g,out

def symmetry_params(rng): return {'axis':rng.choice(['h','v','d1','d2'])}
def symmetry_instance(p,rng):
    axis=p['axis']
    if axis in ('d1','d2'): h=w=rng.randint(4,6)
    else: h,w=rng.randint(4,7),rng.randint(4,7)
    full=empty_grid(h,w); colors=rng.sample(range(1,6),rng.randint(2,4))
    for _ in range(rng.randint(4,9)):
        r,c=rng.randint(0,h-1),rng.randint(0,w-1); v=rng.choice(colors); full[r][c]=v
        if axis=='h': full[h-1-r][c]=v
        elif axis=='v': full[r][w-1-c]=v
        elif axis=='d1': full[c][r]=v
        elif axis=='d2': full[h-1-c][w-1-r]=v
    inp=grid_copy(full); nz=[(r,c) for r in range(h) for c in range(w) if full[r][c]!=0]
    rng.shuffle(nz); n_remove=min(len(nz)//2,rng.randint(2,5))
    for r,c in nz[:n_remove]: inp[r][c]=0
    return inp,full

def region_color_params(rng): return {}
def region_color_instance(p,rng):
    h,w=rng.randint(6,9),rng.randint(6,9); g=empty_grid(h,w,0)
    for _ in range(rng.randint(2,4)):
        if rng.random()<0.5:
            r=rng.randint(1,h-2); c1,c2=sorted(rng.sample(range(w),2))
            for c in range(c1,c2+1): g[r][c]=1
        else:
            c=rng.randint(1,w-2); r1,r2=sorted(rng.sample(range(h),2))
            for r in range(r1,r2+1): g[r][c]=1
    visited=empty_grid(h,w,False); regions=[]
    for r in range(h):
        for c in range(w):
            if g[r][c]==0 and not visited[r][c]:
                cells=[]; q=deque([(r,c)]); visited[r][c]=True
                while q:
                    cr,cc=q.popleft(); cells.append((cr,cc))
                    for dr,dc in [(-1,0),(1,0),(0,-1),(0,1)]:
                        nr,nc=cr+dr,cc+dc
                        if 0<=nr<h and 0<=nc<w and not visited[nr][nc] and g[nr][nc]==0:
                            visited[nr][nc]=True; q.append((nr,nc))
                regions.append(cells)
    regions.sort(key=len); out=grid_copy(g)
    for i,cells in enumerate(regions):
        color=min(i+2,9)
        for r,c in cells: out[r][c]=color
    return g,out

def largest_obj_params(rng): return {}
def largest_obj_instance(p,rng):
    h,w=rng.randint(4,6),rng.randint(4,6); g=empty_grid(h,w)
    for _ in range(rng.randint(2,4)):
        color=rng.randint(1,5); oh,ow=rng.randint(1,3),rng.randint(1,3)
        r0,c0=rng.randint(0,h-oh),rng.randint(0,w-ow)
        for dr in range(oh):
            for dc in range(ow):
                if rng.random()<0.7: g[r0+dr][c0+dc]=color
    comps=connected_components(g)
    if not comps: g[rng.randint(0,h-1)][rng.randint(0,w-1)]=2; comps=connected_components(g)
    largest=max(comps,key=lambda x:x['size'])
    rmin,cmin,rmax,cmax=bounding_box(largest['cells'])
    oh=rmax-rmin+1; ow=cmax-cmin+1; color=largest['color']
    pattern=empty_grid(oh,ow,0)
    for r,c in largest['cells']: pattern[r-rmin][c-cmin]=color
    n_colors=len(distinct_colors(g)); n_rep=max(n_colors,1)
    out_w=ow*n_rep+(n_rep-1); out=empty_grid(oh,out_w,0)
    for rep in range(n_rep):
        off=rep*(ow+1)
        for r in range(oh):
            for c in range(ow): out[r][off+c]=pattern[r][c]
    return g,out

def tile_pattern_params(rng): return {'tile_h':rng.randint(2,3),'tile_w':rng.randint(2,3)}
def tile_pattern_instance(p,rng):
    th,tw=p['tile_h'],p['tile_w']; rh,rw=rng.randint(2,3),rng.randint(2,3)
    h,w=th*rh,tw*rw; tile=empty_grid(th,tw,0)
    for r in range(th):
        for c in range(tw): tile[r][c]=rng.choice([0,0,rng.randint(1,5)])
    full=empty_grid(h,w,0)
    for r in range(h):
        for c in range(w): full[r][c]=tile[r%th][c%tw]
    inp=empty_grid(h,w,0)
    for r in range(th):
        for c in range(tw): inp[r][c]=full[r][c]
    for c in range(min(tw,w)):
        if th<h: inp[th][c]=full[th][c]
    for r in range(min(th,h)):
        if tw<w: inp[r][tw]=full[r][tw]
    return inp,full

def color_chain_params(rng):
    colors=rng.sample(range(1,6),rng.randint(3,4)); chain=colors[:]+[colors[0]]
    return {'mapping':{colors[i]:chain[i+1] for i in range(len(colors))}}
def color_chain_instance(p,rng):
    m=p['mapping']; h,w=rng.randint(4,7),rng.randint(4,7); g=empty_grid(h,w)
    for _ in range(rng.randint(5,h*w//2)): g[rng.randint(0,h-1)][rng.randint(0,w-1)]=rng.choice(list(m.keys()))
    out=empty_grid(h,w)
    for r in range(h):
        for c in range(w): out[r][c]=m.get(g[r][c],0)
    return g,out

def block_expansion_params(rng): return {}
def block_expansion_instance(p,rng):
    h,w=rng.randint(3,5),rng.randint(3,5); g=empty_grid(h,w)
    for _ in range(rng.randint(2,5)): g[rng.randint(0,h-1)][rng.randint(0,w-1)]=rng.randint(2,3)
    nz=[(r,c) for r in range(h) for c in range(w) if g[r][c]!=0]
    scale=max(g[r][c] for r,c in nz) if nz else 2
    out=empty_grid(h*scale,w*scale,0)
    for r in range(h):
        for c in range(w):
            v=g[r][c]
            if v!=0:
                for dr in range(v):
                    for dc in range(v): out[r*scale+dr][c*scale+dc]=v
    return g,out

def conditional_transform_params(rng): return {}
def conditional_transform_instance(p,rng):
    h,w=rng.randint(4,7),rng.randint(4,7); g=empty_grid(h,w)
    for _ in range(rng.randint(1,5)):
        color=rng.randint(1,5); r,c=rng.randint(0,h-1),rng.randint(0,w-1); g[r][c]=color
        if rng.random()<0.4:
            dr,dc=rng.choice([(-1,0),(1,0),(0,-1),(0,1)]); nr,nc=r+dr,c+dc
            if 0<=nr<h and 0<=nc<w: g[nr][nc]=color
    n=len(connected_components(g))
    if n==0: out=[row[::-1] for row in g[::-1]]
    elif n%2==1: out=[row[::-1] for row in g]
    else: out=g[::-1]
    return g,out

def diagonal_fill_params(rng): return {}
def diagonal_fill_instance(p,rng):
    h,w=rng.randint(5,8),rng.randint(5,8); g=empty_grid(h,w)
    for _ in range(rng.randint(2,4)): g[rng.randint(0,h-1)][rng.randint(0,w-1)]=rng.randint(1,5)
    out=grid_copy(g)
    for r in range(h):
        for c in range(w):
            v=g[r][c]
            if v!=0:
                d=r-c
                for i in range(max(0,d),min(h,w+d)):
                    rr,cc=i,i-d
                    if 0<=cc<w and (rr,cc)!=(r,c): out[rr][cc]=v
                s=r+c
                for i in range(max(0,s-w+1),min(h,s+1)):
                    rr,cc=i,s-i
                    if 0<=cc<w and (rr,cc)!=(r,c): out[rr][cc]=v
    return g,out

def row_extremes_params(rng): return {}
def row_extremes_instance(p,rng):
    h,w=rng.randint(4,7),rng.randint(5,8); g=empty_grid(h,w)
    for r in range(h):
        if rng.random()<0.7:
            cols=rng.sample(range(w),min(rng.randint(1,4),w))
            for c in cols: g[r][c]=rng.randint(1,5)
    out=empty_grid(h,w,0)
    for r in range(h):
        nz=[c for c in range(w) if g[r][c]!=0]
        if nz:
            out[r][nz[0]]=8
            if len(nz)>1: out[r][nz[-1]]=9
    return g,out

def shape_sort_params(rng): return {}
def shape_sort_instance(p,rng):
    h,w=rng.randint(5,7),rng.randint(5,7); g=empty_grid(h,w); n_obj=rng.randint(2,4)
    for _ in range(30):
        temp=grid_copy(g); color=rng.randint(1,5); oh,ow=rng.randint(1,3),rng.randint(1,3)
        r0,c0=rng.randint(0,h-oh),rng.randint(0,w-ow); ok=True; cells=[]
        for dr in range(oh):
            for dc in range(ow):
                if rng.random()<0.8:
                    if temp[r0+dr][c0+dc]!=0: ok=False
                    cells.append((r0+dr,c0+dc))
        if ok and cells:
            for r,c in cells: temp[r][c]=color
            g=temp; n_obj-=1
            if n_obj<=0: break
    comps=connected_components(g)
    if len(comps)<2:
        for _ in range(10):
            r,c=rng.randint(0,h-1),rng.randint(0,w-1)
            if g[r][c]==0: g[r][c]=rng.randint(1,5); break
        comps=connected_components(g)
    comps.sort(key=lambda x:x['size']); parts=[]
    for comp in comps:
        rmin,cmin,rmax,cmax=bounding_box(comp['cells'])
        oh=rmax-rmin+1; ow=cmax-cmin+1; obj=empty_grid(oh,ow,0)
        for r,c in comp['cells']: obj[r-rmin][c-cmin]=comp['color']
        parts.append(obj)
    max_h=max(grid_dims(p_)[0] for p_ in parts)
    total_w=sum(grid_dims(p_)[1] for p_ in parts)+len(parts)-1
    out=empty_grid(max_h,total_w,0); off=0
    for p_ in parts:
        ph,pw=grid_dims(p_)
        for r in range(ph):
            for c in range(pw): out[r][off+c]=p_[r][c]
        off+=pw+1
    return g,out

BENCH_TYPES = [
    {'name':'gravity_sort','desc':'Gravity + Column Sort','gen_params':gravity_sort_params,'gen_instance':gravity_sort_instance},
    {'name':'maze_path','desc':'Maze Path (BFS)','gen_params':maze_path_params,'gen_instance':maze_path_instance},
    {'name':'symmetry_completion','desc':'Symmetry Completion','gen_params':symmetry_params,'gen_instance':symmetry_instance},
    {'name':'region_coloring','desc':'Region Flood Coloring','gen_params':region_color_params,'gen_instance':region_color_instance},
    {'name':'largest_replication','desc':'Largest Object Replication','gen_params':largest_obj_params,'gen_instance':largest_obj_instance},
    {'name':'tile_extrapolation','desc':'Tile Pattern Extrapolation','gen_params':tile_pattern_params,'gen_instance':tile_pattern_instance},
    {'name':'color_chain','desc':'Color Chain Transform','gen_params':color_chain_params,'gen_instance':color_chain_instance},
    {'name':'block_expansion','desc':'Block Expansion','gen_params':block_expansion_params,'gen_instance':block_expansion_instance},
    {'name':'conditional_transform','desc':'Conditional Transform','gen_params':conditional_transform_params,'gen_instance':conditional_transform_instance},
    {'name':'diagonal_fill','desc':'Diagonal Fill','gen_params':diagonal_fill_params,'gen_instance':diagonal_fill_instance},
    {'name':'row_extremes','desc':'Row Extremes Marking','gen_params':row_extremes_params,'gen_instance':row_extremes_instance},
    {'name':'shape_sort','desc':'Shape Sort & Arrange','gen_params':shape_sort_params,'gen_instance':shape_sort_instance},
]
print(f"Loaded {len(BENCH_TYPES)} BENCHMARK (transfer) puzzle types.")

def generate_puzzle(ptype, rng, n_examples=3):
    params = ptype['gen_params'](rng); examples = []
    for _ in range(n_examples):
        inp,out = ptype['gen_instance'](params, rng); examples.append({'input':inp,'output':out})
    ti,to = ptype['gen_instance'](params, rng)
    return {'type':ptype['name'],'description':ptype['desc'],'examples':examples,'test_input':ti,'test_output':to}
print("Puzzle generators loaded.")


## 4. RL Environment (ported from `il_rl/env.py`)

`signal_accuracy` scores ONLY non-background cells — the key fix that creates
GRPO variance. On sparse ARC grids (~80% background), `cell_accuracy` makes all
rollouts score 0.8+ → zero GRPO advantage. `signal_accuracy` creates real
variance between rollouts (30% vs 70%), which is what GRPO needs.


In [ ]:
def parse_grid(text):
    last_2d=None; i=0
    while i<len(text):
        if text[i]=='[':
            depth=0
            for j in range(i,len(text)):
                if text[j]=='[': depth+=1
                elif text[j]==']':
                    depth-=1
                    if depth==0:
                        cand=text[i:j+1]
                        if cand.count('[')>=3: last_2d=cand
                        i=j; break
            else: break
        i+=1
    if last_2d:
        rows=re.findall(r'\[\s*([\d\s,]+)\]',last_2d); grid=[]
        for rs in rows:
            nums=re.findall(r'\d+',rs)
            if nums: grid.append([int(x) for x in nums])
        if grid: return grid
    row_matches=re.findall(r'\[\s*\d+[\s,\d]*\]',text)
    if row_matches:
        grid=[]
        for rs in row_matches:
            nums=re.findall(r'\d+',rs)
            if nums: grid.append([int(x) for x in nums])
        if grid: return grid
    return None

def grids_equal(g1,g2):
    if g1 is None or g2 is None: return False
    if len(g1)!=len(g2): return False
    for r1,r2 in zip(g1,g2):
        if len(r1)!=len(r2) or r1!=r2: return False
    return True

def cell_accuracy(pred,target):
    if pred is None or target is None: return 0.0
    if len(pred)!=len(target) or len(pred)==0 or len(pred[0])==0: return 0.0
    h,w=len(target),len(target[0])
    correct=sum(1 for r in range(min(len(pred),h)) for c in range(min(len(pred[r]),w))
                if r<len(target) and c<len(target[r]) and pred[r][c]==target[r][c])
    return correct/(h*w) if h*w>0 else 0.0

def signal_accuracy(pred,target):
    if pred is None or target is None or len(pred)==0 or len(target)==0: return 0.0
    signal=[(r,c,target[r][c]) for r in range(len(target)) for c in range(len(target[r])) if target[r][c]!=0]
    if not signal: return cell_accuracy(pred,target)
    correct=sum(1 for r,c,v in signal if r<len(pred) and c<len(pred[r]) and pred[r][c]==v)
    return correct/len(signal)

def is_degenerate(pred):
    if pred is None or len(pred)==0: return True
    if len(pred)==1 and len(pred[0])<=1: return True
    return all(all(cell==0 for cell in row) for row in pred)

def shape_distance(pred,target):
    if pred is None or len(pred)==0: return 0.0
    eh,ew=len(target),len(target[0]); ph=len(pred); pw=len(pred[0]) if ph>0 else 0
    h_sim=1.0-abs(ph-eh)/max(eh,ph,1); w_sim=1.0-abs(pw-ew)/max(ew,pw,1)
    return (h_sim+w_sim)/2

class RLEnvironment:
    def __init__(self, puzzle, n_steps=3, gamma=0.9):
        self.puzzle=puzzle; self.n_steps=n_steps; self.gamma=gamma
        self.test_output=puzzle['test_output']; self.test_input=puzzle['test_input']
        self.examples=puzzle['examples']; self.step=0; self.prev_quality=0.0
        self.best_accuracy=0.0; self.first_correct_step=None; self.history=[]

    def build_initial_prompt(self):
        lines=["You are an abstract reasoning system. You will be given example input-output grid pairs that demonstrate an unknown transformation rule. You must figure out the rule yourself.",
               "","Grids are 2D arrays of integers 0-9. 0 represents empty/background.","","=== EXAMPLES ===",""]
        for i,ex in enumerate(self.examples):
            lines+=[f"--- Example {i+1} ---","Input:",grid_to_str(ex['input']),"Output:",grid_to_str(ex['output']),""]
        lines+=["=== TEST ===","","Input:",grid_to_str(self.test_input),"",
                "Look at the examples, figure out the transformation rule, and predict the test output grid.",
                "Output your reasoning followed by the predicted grid as a 2D array."]
        return '\n'.join(lines)

    def build_feedback_prompt(self, step, accuracy, predicted_grid):
        h,w=grid_dims(self.test_output); total=h*w; correct=int(accuracy*total)
        lines=[f"FEEDBACK: Your prediction has {accuracy*100:.0f}% cell accuracy ({correct}/{total} cells correct)."]
        if accuracy==1.0: lines.append("Your prediction is exactly correct!")
        elif accuracy>0: lines.append("Your prediction is partially correct. Some cells are wrong.")
        else: lines.append("Your prediction does not match the expected output at all.")
        if predicted_grid is not None:
            ph=len(predicted_grid); pw=len(predicted_grid[0]) if predicted_grid else 0
            eh,ew=grid_dims(self.test_output)
            if (ph,pw)!=(eh,ew): lines.append(f"Hint: the output grid has dimensions {eh}x{ew}, but your prediction was {ph}x{pw}.")
        if step<self.n_steps-1:
            lines+=["","Refine your understanding and predict again.","Output your reasoning followed by the predicted grid as a 2D array."]
        else:
            lines+=["","This is your final attempt. Make your best prediction.","Output your reasoning followed by the predicted grid as a 2D array."]
        return '\n'.join(lines)

    def process_action(self, generated_text):
        pred=parse_grid(generated_text); acc=cell_accuracy(pred,self.test_output)
        exact=grids_equal(pred,self.test_output)
        exp_colors=set(c for row in self.test_output for c in row if c!=0)
        pred_colors=set(c for row in pred for c in row if c!=0) if pred else set()
        color_overlap=len(pred_colors&exp_colors)/max(len(exp_colors),1)
        quality=signal_accuracy(pred,self.test_output)+0.15*shape_distance(pred,self.test_output)+0.05*color_overlap
        if is_degenerate(pred): quality*=0.3
        quality=min(quality,1.0)
        process_reward=(self.gamma**self.step)*quality
        if self.step>0 and quality>self.prev_quality:
            process_reward+=(self.gamma**self.step)*(quality-self.prev_quality)*0.5
        if exact and self.first_correct_step is None: self.first_correct_step=self.step
        terminal=0.0; done=(self.step>=self.n_steps-1) or exact
        if exact: terminal=(self.gamma**self.step)*3.0
        reward=process_reward+terminal
        self.history.append({'step':self.step,'accuracy':acc,'quality':quality,'exact':exact,'reward':reward})
        self.prev_quality=quality; self.best_accuracy=max(self.best_accuracy,acc); self.step+=1
        return reward,acc,pred,done

    def total_reward(self): return sum(h['reward'] for h in self.history)
print("RL environment loaded.")


## 5. Fast Rollout Collector (uses `model.generate()` with KV cache)

**Key speed fix**: use HuggingFace's optimized `model.generate()` which
internally manages the KV cache (O(n²) total) instead of a manual token loop
that recomputes the full sequence every step (O(n³)).

For GRPO we need behavior-policy logprobs (`old_logprobs`). We get these from
`output_scores=True` in `model.generate()` — the scores are the pre-sampling
logit distributions, and we compute log_softmax at the generated token positions.

For the GRPO update (with gradients), we do a **single forward pass** over the
full episode to get new logprobs at action positions (the `compute_action_logprobs`
function, ported from the repo's MLX version with the same memory optimization:
apply LM head only at action-token positions).


In [ ]:
THINK_CLOSE_ID = tokenizer.convert_tokens_to_ids("")
THINK_CLOSE_STR = ""
print(f" token id: {THINK_CLOSE_ID}")

@torch.no_grad()
def generate_response(prompt_ids, max_new_tokens, temperature, top_p, seed=None):
    """Generate a response using model.generate() with KV cache.

    Returns (generated_ids, logprobs) where logprobs[t] is the log-prob of
    generated_ids[t] under the behavior policy (from output_scores).
    """
    if seed is not None:
        torch.manual_seed(seed)
    input_ids = torch.tensor([prompt_ids], dtype=torch.long, device=DEVICE)
    do_sample = temperature > 0
    out = model.generate(
        input_ids,
        max_new_tokens=max_new_tokens,
        do_sample=do_sample,
        temperature=temperature if do_sample else 1.0,
        top_p=top_p if do_sample else 1.0,
        pad_token_id=tokenizer.eos_token_id,
        return_dict_in_generate=True,
        output_scores=True,
    )
    gen_ids = out.sequences[0][input_ids.shape[1]:].tolist()
    # Compute logprobs from scores (scores[t] = logits before sampling token t)
    logprobs = []
    for t, tid in enumerate(gen_ids):
        logits = out.scores[t][0].float()
        lp = F.log_softmax(logits, dim=-1)[tid].item()
        logprobs.append(lp)
    return gen_ids, logprobs

def collect_rollout(puzzle, n_steps, gamma, thinking_tokens, prediction_tokens,
                    temperature, top_p, seed=None):
    """Collect one multi-step rollout using fast model.generate().

    Returns dict with tokens, action_positions, old_logprobs, reward, env.
    """
    env = RLEnvironment(puzzle, n_steps=n_steps, gamma=gamma)
    all_tokens = []; action_positions = []; old_logprobs = []
    msgs = []; prev_acc = 0.0; prev_grid = None

    for step in range(n_steps):
        if step == 0:
            msgs = [{'role':'user','content': env.build_initial_prompt()}]
        else:
            msgs.append({'role':'user','content': env.build_feedback_prompt(step, prev_acc, prev_grid)})

        prompt_ids = tokenizer.apply_chat_template(msgs, tokenize=True, add_generation_prompt=True)
        # For multi-step: extend all_tokens with the new prompt portion
        new_prompt = prompt_ids[len(all_tokens):]
        all_tokens.extend(new_prompt)
        gen_start = len(all_tokens)

        # Generate thinking + answer in one call (model emits </midt> naturally
        # if it converges; parse_grid finds the 2D array regardless)
        total_tokens = thinking_tokens + prediction_tokens
        gen_ids, gen_lp = generate_response(prompt_ids, total_tokens, temperature, top_p, seed=seed)
        all_tokens.extend(gen_ids)
        old_logprobs.extend(gen_lp)
        gen_end = len(all_tokens)
        if gen_end > gen_start:
            action_positions.append((gen_start, gen_end))

        generated_text = tokenizer.decode(gen_ids, skip_special_tokens=False)
        eos_tok = tokenizer.eos_token
        if eos_tok and eos_tok in generated_text:
            generated_text = generated_text.replace(eos_tok, '').strip()
        reward, acc, grid, done = env.process_action(generated_text)
        msgs.append({'role':'assistant','content': generated_text})
        prev_acc = acc; prev_grid = grid
        if done: break

    return {'tokens': all_tokens, 'action_positions': action_positions,
            'old_logprobs': old_logprobs, 'reward': env.total_reward(),
            'env': env, 'messages': msgs, 'n_steps': len(action_positions)}

print("Fast rollout collector loaded.")


## 6. GRPO Trainer

**Algorithm**: collect G rollouts → group-relative advantages → PPO-clipped
policy gradient + KL penalty → update LoRA params.

The GRPO update does a **single forward pass** over each rollout's full episode
to compute new logprobs at action-token positions (with gradients). The LM head
is applied only at action positions (the memory fix from the repo).


In [ ]:
def compute_action_logprobs(tokens, action_positions):
    """Forward the full episode, return logprobs at action positions only.
    Memory optimization: gather logprobs only at action-token positions."""
    input_ids = torch.tensor([tokens[:-1]], dtype=torch.long, device=DEVICE)
    out = model(input_ids, use_cache=False)
    logits = out.logits[0].float()  # [seq_len-1, vocab]
    segments = []
    for (start, end) in action_positions:
        lp_start = start - 1; lp_end = end - 1
        if lp_end <= lp_start: continue
        seg_logits = logits[lp_start:lp_end, :]
        seg_logprobs = F.log_softmax(seg_logits, dim=-1)
        seg_tokens = torch.tensor(tokens[start:end], dtype=torch.long, device=DEVICE)
        seg_lp = seg_logprobs.gather(1, seg_tokens.unsqueeze(1)).squeeze(1)
        segments.append(seg_lp)
    return segments

def grpo_loss_for_rollout(rollout, advantage, clip_eps, kl_beta):
    tokens = rollout['tokens']; action_positions = rollout['action_positions']
    old_lp = torch.tensor(rollout['old_logprobs'], dtype=torch.float32, device=DEVICE)
    new_segments = compute_action_logprobs(tokens, action_positions)
    if not new_segments: return None, 0
    new_lp = torch.cat(new_segments)
    n = min(len(new_lp), len(old_lp))
    new_lp = new_lp[:n]; old_lp = old_lp[:n]
    ratio = torch.exp(new_lp - old_lp)
    clipped = torch.clamp(ratio, 1-clip_eps, 1+clip_eps)
    pg_loss = -torch.min(ratio * advantage, clipped * advantage)
    kl = (new_lp - old_lp).mean()
    return pg_loss.mean() + kl_beta * kl, n

def compute_advantages(rollouts, eps=1e-8):
    rewards = np.array([r['reward'] for r in rollouts])
    mean_r = rewards.mean(); std_r = rewards.std()
    if std_r < eps: return [0.0]*len(rollouts), float(mean_r), float(std_r)
    return ((rewards - mean_r) / (std_r + eps)).tolist(), float(mean_r), float(std_r)

class GRPOTrainer:
    def __init__(self, lr, clip_eps, group_size, gamma, n_steps,
                 thinking_tokens, prediction_tokens, temperature, top_p, kl_beta):
        self.lr=lr; self.clip_eps=clip_eps; self.group_size=group_size
        self.gamma=gamma; self.n_steps=n_steps
        self.thinking_tokens=thinking_tokens; self.prediction_tokens=prediction_tokens
        self.temperature=temperature; self.top_p=top_p; self.kl_beta=kl_beta
        self.iteration=0
        self.optimizer = torch.optim.AdamW(
            [p for p in model.parameters() if p.requires_grad], lr=lr)

    def train_step(self, puzzle):
        t0 = time.time(); torch.cuda.empty_cache()
        # Phase 1: collect G rollouts (no grad, fast generate)
        model.eval(); rollouts = []
        for g in range(self.group_size):
            seed = 42 + self.iteration*1000 + g*10000
            r = collect_rollout(puzzle, self.n_steps, self.gamma,
                                self.thinking_tokens, self.prediction_tokens,
                                self.temperature, self.top_p, seed=seed)
            rollouts.append(r); torch.cuda.empty_cache()
        rollout_time = time.time() - t0
        # Phase 2: advantages
        advantages, mean_r, std_r = compute_advantages(rollouts)
        # Phase 3: GRPO update (with grad, single forward pass per rollout)
        model.train(); t1 = time.time()
        loss_sum = 0.0; n_updated = 0
        self.optimizer.zero_grad()
        for rollout, adv in zip(rollouts, advantages):
            if abs(adv) < 1e-8: continue
            loss, n_tok = grpo_loss_for_rollout(rollout, adv, self.clip_eps, self.kl_beta)
            if loss is None or n_tok == 0: continue
            (loss / self.group_size).backward()
            loss_sum += float(loss.item()); n_updated += 1
            torch.cuda.empty_cache()
        if n_updated > 0:
            torch.nn.utils.clip_grad_norm_(
                [p for p in model.parameters() if p.requires_grad], 1.0)
            self.optimizer.step()
        torch.cuda.empty_cache()
        update_time = time.time() - t1
        rewards = [r['reward'] for r in rollouts]
        best_acc = max(r['env'].best_accuracy for r in rollouts)
        any_exact = any(r['env'].first_correct_step is not None for r in rollouts)
        avg_toks = float(np.mean([len(r['tokens']) for r in rollouts]))
        peak = torch.cuda.max_memory_allocated()/1e9
        torch.cuda.reset_peak_memory()
        m = {'iteration':self.iteration,'mean_reward':mean_r,'std_reward':std_r,
             'max_reward':max(rewards),'min_reward':min(rewards),
             'best_accuracy':best_acc,'any_exact':any_exact,
             'advantages':advantages,'avg_episode_tokens':avg_toks,
             'rollout_time':rollout_time,'update_time':update_time,
             'peak_memory':peak,'loss':loss_sum/max(n_updated,1)}
        self.iteration += 1
        return m

print("GRPO trainer loaded.")


## 7. Build Training Sampler + 40-Task Benchmark

In [ ]:
def build_puzzle_sampler(seed):
    rng = random.Random(seed); types = list(TRAIN_TYPES); rng.shuffle(types); idx=[0]
    def next_puzzle():
        et = types[idx[0]%len(types)]; idx[0]+=1
        p = generate_puzzle(et, rng); p['id'] = f"{et['name']}_{rng.randint(0,99999)}"
        return p
    return next_puzzle

def build_benchmark(seed, total=40):
    rng = random.Random(seed + 500000)
    types = list(BENCH_TYPES); puzzles = []
    for i in range(total):
        et = types[i%len(types)]
        p = generate_puzzle(et, rng); p['id'] = f"bench_{et['name']}_{i:03d}"
        puzzles.append(p)
    return puzzles

next_puzzle = build_puzzle_sampler(SEED)
benchmark_puzzles = build_benchmark(SEED, BENCHMARK_SIZE)
print(f"Benchmark: {len(benchmark_puzzles)} puzzles from {len(BENCH_TYPES)} transfer types")
print(f"  types: {sorted(set(p['type'] for p in benchmark_puzzles))}")
# Verify no overlap with training types
train_names = set(t['name'] for t in TRAIN_TYPES)
bench_names = set(t['name'] for t in BENCH_TYPES)
assert not (train_names & bench_names), "Training and benchmark types overlap!"
print(f"  NO overlap with {len(TRAIN_TYPES)} training types: OK")

with open('/kaggle/working/benchmark_dataset.json','w') as f:
    json.dump([{'id':p['id'],'type':p['type'],'description':p['description'],
                'examples':p['examples'],'test_input':p['test_input'],
                'test_output':p['test_output']} for p in benchmark_puzzles], f, indent=2)
print("Saved benchmark dataset.")


## 8. GRPO Training Loop (40 iterations)

Each iteration: sample a puzzle → collect 4 rollouts → compute group-relative
advantages → PPO-clipped policy gradient + KL penalty update on LoRA params.

No in-training held-out eval (the pre/post 40-task benchmark is the real
comparison and avoids spending time on slow intermediate evals).


In [ ]:
trainer = GRPOTrainer(
    lr=LR, clip_eps=CLIP_EPS, group_size=GROUP_SIZE, gamma=GAMMA,
    n_steps=N_STEPS, thinking_tokens=THINKING_TOKENS,
    prediction_tokens=PREDICTION_TOKENS, temperature=TEMPERATURE,
    top_p=TOP_P, kl_beta=KL_BETA)

metrics_history = []
print(f"Starting GRPO training: {N_TRAIN_ITERS} iters")
print(f"  per-iter: {GROUP_SIZE} rollouts × {N_STEPS} steps × {THINKING_TOKENS+PREDICTION_TOKENS} tokens")
print(f"  est. ~{GROUP_SIZE*N_STEPS*(THINKING_TOKENS+PREDICTION_TOKENS)} tokens/iter\n")

overall_t0 = time.time()
for it in range(N_TRAIN_ITERS):
    puzzle = next_puzzle()
    try:
        m = trainer.train_step(puzzle)
    except Exception as e:
        print(f"[iter {it}] ERROR on {puzzle.get('id','?')}: {e}")
        print(traceback.format_exc()[-500:])
        torch.cuda.empty_cache(); continue
    metrics_history.append(m)
    print(f"[iter {m['iteration']:3d}] {puzzle.get('id','?'):<22} "
          f"R={m['mean_reward']:+.3f} (std {m['std_reward']:.3f}, max {m['max_reward']:+.3f}) "
          f"best_acc={m['best_accuracy']:.2f} exact={m['any_exact']} "
          f"loss={m['loss']:+.4f} toks={m['avg_episode_tokens']:.0f} "
          f"mem={m['peak_memory']:.2f}GB "
          f"t={m['rollout_time']:.0f}+{m['update_time']:.0f}s")
    # Save best LoRA if this iter had any exact match (simple checkpoint policy)
    if m['any_exact']:
        model.save_pretrained("/kaggle/working/best_lora")

elapsed = time.time() - overall_t0
print(f"\n{'='*70}")
print(f"RL training complete: {len(metrics_history)} iters in {elapsed:.0f}s ({elapsed/60:.1f} min)")
print(f"{'='*70}")
if metrics_history:
    print(f"  train mean R: first {metrics_history[0]['mean_reward']:+.3f} -> last {metrics_history[-1]['mean_reward']:+.3f}")
    exacts = sum(1 for m in metrics_history if m['any_exact'])
    print(f"  iters with exact rollout: {exacts}/{len(metrics_history)}")
    print(f"  peak mem: {max(m['peak_memory'] for m in metrics_history):.2f} GB")
# Always save final adapter
model.save_pretrained("/kaggle/working/final_lora")
with open('/kaggle/working/train_metrics.json','w') as f: json.dump(metrics_history, f, indent=2)
print("Saved final_lora/ and train_metrics.json")


## 9. Benchmark: Pre-RL vs Post-RL (40 transfer tasks)

Run the 40-task benchmark **twice**:
1. **Pre-RL**: base model with LoRA adapters **disabled** (greedy, deterministic)
2. **Post-RL**: with trained LoRA adapters **enabled** (greedy, deterministic)

Greedy decoding (temperature=0) for reproducibility. Single rollout per puzzle.


In [ ]:
def shape_match(g1,g2):
    if g1 is None or g2 is None: return False
    return grid_dims(g1)==grid_dims(g2)

def cell_acc(g1,g2):
    if g1 is None or g2 is None or not shape_match(g1,g2): return 0.0
    h,w=grid_dims(g1)
    if h==0 or w==0: return 0.0
    return sum(1 for r in range(h) for c in range(w) if g1[r][c]==g2[r][c])/(h*w)

@torch.no_grad()
def benchmark_run(puzzles, think_tokens, answer_tokens, label=""):
    """Greedy inference on each puzzle. Returns list of result dicts."""
    model.eval(); results = []; t0 = time.time()
    for i, p in enumerate(puzzles):
        torch.cuda.empty_cache()
        try:
            # Single greedy rollout (n_steps=1 for benchmark — one shot, no feedback)
            rollout = collect_rollout(p, n_steps=1, gamma=1.0,
                                      thinking_tokens=think_tokens,
                                      prediction_tokens=answer_tokens,
                                      temperature=0.0, top_p=1.0, seed=0)
            env = rollout['env']
            # Get the predicted grid from the last assistant message
            pred = None
            if rollout['messages']:
                ans = rollout['messages'][-1]['content']
                if THINK_CLOSE_STR in ans: ans = ans.split(THINK_CLOSE_STR, 1)[1]
                pred = parse_grid(ans)
            em = grids_equal(pred, p['test_output'])
            sm = shape_match(pred, p['test_output'])
            ca = cell_acc(pred, p['test_output'])
            results.append({'puzzle_id':p['id'],'puzzle_type':p['type'],
                            'description':p['description'],'exact_match':em,
                            'shape_match':sm,'cell_accuracy':ca,
                            'has_prediction':pred is not None,
                            'predicted_dims':str(grid_dims(pred) if pred else None),
                            'expected_dims':str(grid_dims(p['test_output']))})
            status = "EXACT" if em else ("SHAPE" if sm else "MISS")
            print(f"  [{label} {i+1:2d}/{len(puzzles)}] {p['id']:<28} {status:5s} "
                  f"acc={ca:.2%} dims={grid_dims(pred) if pred else None}/{grid_dims(p['test_output'])}")
        except Exception as e:
            print(f"  [{label} {i+1:2d}/{len(puzzles)}] {p['id']}: ERROR {str(e)[:100]}")
            results.append({'puzzle_id':p['id'],'puzzle_type':p['type'],
                            'description':p['description'],'exact_match':False,
                            'shape_match':False,'cell_accuracy':0.0,
                            'has_prediction':False,'predicted_dims':'None',
                            'expected_dims':str(grid_dims(p['test_output']))})
    elapsed = time.time() - t0
    print(f"  {label} done: {len(results)} puzzles in {elapsed:.0f}s ({elapsed/60:.1f} min)")
    return results

print("="*70)
print("PRE-RL BENCHMARK (base model, LoRA disabled)")
print("="*70)
model.disable_adapter_layers()
pre_results = benchmark_run(benchmark_puzzles, BENCH_THINK, BENCH_ANSWER, label="pre")
model.enable_adapter_layers()

print(f"\n{'='*70}")
print("POST-RL BENCHMARK (trained LoRA enabled)")
print("="*70)
post_results = benchmark_run(benchmark_puzzles, BENCH_THINK, BENCH_ANSWER, label="post")

with open('/kaggle/working/benchmark_pre_rl.json','w') as f: json.dump(pre_results, f, indent=2)
with open('/kaggle/working/benchmark_post_rl.json','w') as f: json.dump(post_results, f, indent=2)
print("\nSaved pre/post benchmark results.")


## 10. Results: Pre-RL vs Post-RL Comparison

In [ ]:
def summarize(results, label):
    total=len(results); exact=sum(1 for r in results if r['exact_match'])
    shape=sum(1 for r in results if r['shape_match'])
    avg_cell=float(np.mean([r['cell_accuracy'] for r in results])) if results else 0.0
    has_pred=sum(1 for r in results if r['has_prediction'])
    print(f"\n{'='*70}\n{label}: {total} puzzles\n{'='*70}")
    print(f"  Exact matches: {exact}/{total} = {exact/total:.1%}")
    print(f"  Shape matches: {shape}/{total} = {shape/total:.1%}")
    print(f"  Avg cell acc:  {avg_cell:.1%}")
    print(f"  Has prediction: {has_pred}/{total}")
    print(f"\n  {'Puzzle Type':<28} {'Exact':>7} {'Shape':>7} {'CellAcc':>8}")
    print(f"  {'-'*52}")
    by_type={}
    for r in results: by_type.setdefault(r['puzzle_type'],[]).append(r)
    for t in sorted(by_type):
        rs=by_type[t]; em=sum(1 for r in rs if r['exact_match'])
        sm=sum(1 for r in rs if r['shape_match']); ca=float(np.mean([r['cell_accuracy'] for r in rs]))
        print(f"  {rs[0]['description'][:26]:<28} {em:>3}/{len(rs):<3} {sm:>3}/{len(rs):<3} {ca:>7.1%}")
    return {'total':total,'exact':exact,'shape':shape,'avg_cell':avg_cell,
            'exact_rate':exact/total if total else 0,'shape_rate':shape/total if total else 0}

pre_sum  = summarize(pre_results,  "PRE-RL  (base model)")
post_sum = summarize(post_results, "POST-RL (trained LoRA)")

print(f"\n{'='*70}\nDELTA (post - pre)\n{'='*70}")
print(f"  Exact match rate: {pre_sum['exact_rate']:.1%} -> {post_sum['exact_rate']:.1%}  (Δ {post_sum['exact_rate']-pre_sum['exact_rate']:+.1%})")
print(f"  Shape match rate: {pre_sum['shape_rate']:.1%} -> {post_sum['shape_rate']:.1%}  (Δ {post_sum['shape_rate']-pre_sum['shape_rate']:+.1%})")
print(f"  Avg cell accuracy: {pre_sum['avg_cell']:.1%} -> {post_sum['avg_cell']:.1%}  (Δ {post_sum['avg_cell']-pre_sum['avg_cell']:+.1%})")


## 11. Visualizations

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Training reward curve
if metrics_history:
    iters = [m['iteration'] for m in metrics_history]
    rewards = [m['mean_reward'] for m in metrics_history]
    axes[0,0].plot(iters, rewards, 'o-', color='#06d6a0', markersize=3)
    axes[0,0].set_xlabel('GRPO iteration'); axes[0,0].set_ylabel('Mean reward (per puzzle)')
    axes[0,0].set_title('Training Reward (per-puzzle, not a learning curve)')

# 2. Pre vs Post exact match per type
by_type_pre={}; by_type_post={}
for r in pre_results: by_type_pre.setdefault(r['puzzle_type'],[]).append(r)
for r in post_results: by_type_post.setdefault(r['puzzle_type'],[]).append(r)
types_sorted=sorted(by_type_pre.keys())
em_pre=[sum(1 for r in by_type_pre[t] if r['exact_match'])/len(by_type_pre[t]) for t in types_sorted]
em_post=[sum(1 for r in by_type_post[t] if r['exact_match'])/len(by_type_post[t]) for t in types_sorted]
labels=[by_type_pre[t][0]['description'][:18] for t in types_sorted]
x=np.arange(len(types_sorted)); w=0.4
axes[0,1].barh(x-w/2, em_pre, w, color='#e63946', label='pre-RL')
axes[0,1].barh(x+w/2, em_post, w, color='#06d6a0', label='post-RL')
axes[0,1].set_yticks(x); axes[0,1].set_yticklabels(labels, fontsize=8)
axes[0,1].set_xlabel('Exact match rate'); axes[0,1].set_title('Benchmark: Exact Match per Type')
axes[0,1].legend(); axes[0,1].set_xlim(0,1)

# 3. Cell accuracy per puzzle: pre vs post
ca_pre=[r['cell_accuracy'] for r in pre_results]; ca_post=[r['cell_accuracy'] for r in post_results]
colors=['#06d6a0' if (pre_results[i]['exact_match']==False and post_results[i]['exact_match']==True)
        else ('#118ab2' if post_results[i]['exact_match'] else '#e63946') for i in range(len(pre_results))]
axes[1,0].scatter(ca_pre, ca_post, c=colors, s=60, alpha=0.7)
axes[1,0].plot([0,1],[0,1], 'k--', alpha=0.3)
axes[1,0].set_xlabel('Pre-RL cell accuracy'); axes[1,0].set_ylabel('Post-RL cell accuracy')
axes[1,0].set_title('Per-puzzle: Pre vs Post (green=newly exact)')
axes[1,0].set_xlim(-0.05,1.05); axes[1,0].set_ylim(-0.05,1.05)

# 4. Training loss
if metrics_history:
    losses=[m['loss'] for m in metrics_history]
    axes[1,1].plot(iters, losses, 's-', color='#ef476f', markersize=3)
    axes[1,1].set_xlabel('GRPO iteration'); axes[1,1].set_ylabel('Loss')
    axes[1,1].set_title('GRPO Training Loss')

plt.tight_layout()
plt.savefig('/kaggle/working/results.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved results.png")


## 12. Detailed Per-Puzzle Comparison

In [ ]:
print(f"{'#':<3} {'Puzzle ID':<28} {'Type':<22} {'Pre-EM':>7} {'Post-EM':>8} {'Pre-CA':>7} {'Post-CA':>8} {'Δ':>6}")
print("="*95)
n_improved=0; n_regressed=0
for i in range(len(pre_results)):
    pr,po=pre_results[i],post_results[i]
    em_pre='Y' if pr['exact_match'] else 'n'; em_post='Y' if po['exact_match'] else 'n'
    delta=po['cell_accuracy']-pr['cell_accuracy']; flag=''
    if not pr['exact_match'] and po['exact_match']: flag=' <-- NEWLY EXACT'; n_improved+=1
    elif pr['exact_match'] and not po['exact_match']: flag=' <-- REGRESSED'; n_regressed+=1
    print(f"{i+1:<3} {pr['puzzle_id']:<28} {pr['puzzle_type']:<22} {em_pre:>7} {em_post:>8} "
          f"{pr['cell_accuracy']:>6.1%} {po['cell_accuracy']:>7.1%} {delta:>+5.1%}{flag}")
print("="*95)
print(f"Newly exact after RL: {n_improved} | Regressed: {n_regressed}")


## 13. Final Summary & Artifacts

In [ ]:
import os
print("="*70)
print("FINAL SUMMARY")
print("="*70)
print(f"Model: {MODEL_NAME}")
print(f"LoRA: rank={LORA_RANK}, layers={LORA_LAYERS}")
print(f"GRPO: {N_TRAIN_ITERS} iters, group={GROUP_SIZE}, steps={N_STEPS}, lr={LR}")
print(f"Benchmark: {BENCHMARK_SIZE} transfer puzzles ({len(BENCH_TYPES)} types)")
print(f"")
print(f"PRE-RL:  exact {pre_sum['exact']}/{pre_sum['total']} ({pre_sum['exact_rate']:.1%}), avg cell acc {pre_sum['avg_cell']:.1%}")
print(f"POST-RL: exact {post_sum['exact']}/{post_sum['total']} ({post_sum['exact_rate']:.1%}), avg cell acc {post_sum['avg_cell']:.1%}")
print(f"Δ exact: {post_sum['exact_rate']-pre_sum['exact_rate']:+.1%}  Δ cell acc: {post_sum['avg_cell']-pre_sum['avg_cell']:+.1%}")
print(f"")
print("Artifacts in /kaggle/working/:")
for f in sorted(os.listdir('/kaggle/working')):
    p=os.path.join('/kaggle/working',f)
    if os.path.isfile(p): print(f"  {f} ({os.path.getsize(p)/1024:.0f} KB)")
    else: print(f"  {f}/ (dir)")
print("="*70)


## Notes

### v2 speed fixes
- **`model.generate()` with KV cache**: replaces the manual token loop that
  recomputed full-sequence attention every step (O(n³)). Now O(n²) — ~10x faster.
- **Single GPU**: the 1.5B model (3 GB bf16) runs on `cuda:0`. Pipeline
  parallelism across 2 GPUs adds latency for sequential generation without
  benefit when the model fits on one card.
- **`output_scores=True`**: gets behavior-policy logprobs from `model.generate()`
  directly — no separate forward pass needed during rollout collection.
- **Reduced budgets**: think=128, pred=96 (from 256/128). Skipped in-training
  eval (the 40-task pre/post benchmark is the real comparison).

### Why GRPO + signal_accuracy
GRPO uses group-relative advantages (no value network). `signal_accuracy` scores
only non-background cells, creating variance between rollouts on sparse ARC grids
— that variance is the GRPO learning signal.

### Why transfer types for benchmark
The 12 benchmark types are different from the 20 training types. If we
benchmarked on training types, the model could memorize. Transfer types test
whether RL improved general abstract reasoning.
